# Hyperspectral Object Tracking with SAM3
When visually similar people move through the same patch of ground, a conventional RGB tracker loses track: 
appearance cues — colour, texture, shape — are too similar to reliably distinguish identities.
A Cubert Ultris XMR records **61 spectral bands per pixel** (430–910 nm) at 15 Hz — far beyond
what the eye, or any RGB sensor, can resolve. Different fabrics, dyes, and coatings
scatter and absorb light differently across those extra bands, giving each outfit a
distinct *spectral fingerprint* even when the visible-light appearance is identical.

This tutorial builds a Cuvis.AI pipeline that feeds SAM3 (Meta's Segment Anything
Model 3) a **faithful RGB** image synthesised from CIE tristimulus colour matching of
the hyperspectral cube. The tracker sees calibrated colour and can preserve
identities through occlusions that defeat a plain RGB tracker.

**Pipeline at a glance:**

```
CU3SDataNode ──► CIETristimulusRGBSelector ──► SAM3TextPropagation ──► TrackingOverlayNode ──► ToVideoNode
                                                ▲
                                          TextPrompt("person@0")
```

> **Prerequisites**
>
> 1. Install cuvis-ai — see the [Installation Guide](https://docs.cuvis.ai/latest/user-guide/installation/).
> 2. Notebook environment: `uv sync --extra dev`, then `uv run jupyter lab`.

In [ ]:
# ruff: noqa: E402
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Image, Video, display
from loguru import logger

from cuvis_ai.node.anomaly_visualization import TrackingOverlayNode
from cuvis_ai.node.channel_selector import CIETristimulusRGBSelector, CIRSelector
from cuvis_ai.node.data import CU3SDataNode
from cuvis_ai.node.json_file import CocoTrackMaskWriter
from cuvis_ai.node.prompts import MaskPrompt, TextPrompt
from cuvis_ai.node.video import ToVideoNode, VideoFrameDataModule, VideoFrameNode
from cuvis_ai_core.data.datasets import SingleCu3sDataModule
from cuvis_ai_core.data.public_datasets import PublicDatasets
from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.training import Predictor
from cuvis_ai_core.utils.node_registry import NodeRegistry

## 1 · Fetch the dataset

The XMR Object Tracking dataset (~25 GB) lives on Hugging Face Hub. It contains
two recording sessions:

| Session | Folder | Scene |
|---------|--------|-------|
| **Passive** | `2026_04_15_16_28_10` | Three actors in identical outfits walk in single file, occluding one another. The target wears a garment that looks distinct in colour-infrared (CIR) even though it matches in the visible spectrum. |
| **Active** | `2026_04_20_16_28_54` | Midway through, the target is sprayed with "invisible" spectral ink visible only in specific NIR bands. See §12 for this mode. |

This tutorial uses the **passive session** for the main tracking pipeline.

In [ ]:
dataset_dir = Path("../../data")

_ = PublicDatasets.download_dataset(
    "demo_object_tracking",
    download_path=str(dataset_dir),
    force=False,
)

## 2 · The visual ambiguity problem

Tracking a specific person in a crowd is hard when appearance cues
collapse. State-of-the-art video trackers such as SAM3 rely on colour,
texture, and shape to maintain identity across frames. Visual ambiguity
arises whenever those cues become non-discriminative in the RGB domain —
subjects in similar clothing, targets far from the camera, or partial
occlusions that expose only a sliver of the person. When
re-identification confidence drops, the tracker swaps IDs on the next
overlap.

**Why does hyperspectral video help?**

The visible spectrum covers roughly 380–740 nm. Standard RGB cameras
sample three broad bands in this range. The Cubert XMR samples **61
narrow bands from 430 to 910 nm**, extending well into the near-infrared.
In the NIR, many common fabric dyes, coatings, and chemical treatments
behave very differently from one another even when their visible-light
colour is indistinguishable. Two materials that match perfectly at 550 nm
may differ by 30 % or more in reflectance at 800 nm.

**CIE tristimulus** collapses the 61-band cube into three channels by
integrating against the CIE 1931 colour-matching functions ($\bar{x}$,
$\bar{y}$, $\bar{z}$) — the same response curves that define human
colour perception. The result is **true RGB**: what a conventional camera
would photograph if it were pointed at the same scene. This lets SAM3 run
in its native RGB mode without retraining or modification. Other
pipelines in this repository substitute a different selector — CIR or a
custom band combination — to feed the tracker a *false-RGB* view that
surfaces spectral features invisible in true colour.

The result: SAM3 receives a clean, calibrated RGB rendering of every
frame and can preserve target identity through visually ambiguous
conditions that cause an RGB-only tracker to swap.

**A note on tracking mode.** This pipeline uses `SAM3TextPropagation`
with the prompt `"person"` — a *class-level* tracker that segments all
people present. To lock onto one specific target instead, switch to
`SAM3MaskPropagation` and seed it with a single mask from one frame —
shown in §10 (RGB) and §11 (CIR).

## 3 · Tutorial configuration

Edit these variables to customise the run.

- **`PROCESSING_MODE`** — `"SpectralRadiance"` is correct for this dataset: no
  white reference was recorded, so reflectance calibration is not possible.
- **`TEXT_PROMPT`** — SAM3 text prompt injected at frame 0. `"person@0"` seeds a
  class-level tracker that finds all people. Append `@<frame>` to re-seed at a
  later frame, e.g. `"person@30"`.
- **`MASK_PROMPT_SPEC`** — seed for the §10–11 mask propagation, in the form
  `<object_id>:<detection_id>@<frame_id>`. `"1:3@20"` means *"track the third
  detection at frame 20 as object 1"*. The detections come from the COCO JSON
  written by §7.
- **`PREVIEW_FRAME_ID`** — frame index for the §6 sanity-check.
- **`FRAME_RATE`** — output video FPS (matches the sensor capture rate).

§7 caps the text-propagation run at 200 frames (hard-coded in that cell — SAM3
text propagation is the slowest step, so we keep it bounded). §9 (CIR), §10–11
(mask propagation) and §12 (active preview) all run on the full source video.

In [ ]:
PROCESSING_MODE = "SpectralRadiance"
TEXT_PROMPT = "person@0"
MASK_PROMPT_SPEC = "1:3@20"  # assign tracker object_id=1 to detection_id=3 at source frame 20
PREVIEW_FRAME_ID = 50
FRAME_RATE = 15.0

output_dir = Path("./output/object_tracking")
output_dir.mkdir(parents=True, exist_ok=True)

# Outputs of the §5–7 text-propagation pipeline (consumed by §10–11):
output_video_path = output_dir / "tracking_overlay.mp4"  # SAM3 text-prop overlay video
rgb_video_path = output_dir / "rgb_video.mp4"  # clean CIE-tristimulus RGB video
output_json_path = output_dir / "tracking_results.json"  # COCO mask JSON used by MaskPrompt

# Outputs of §9 (CIR rendering) and §10–11 (mask propagation):
cir_video_path = output_dir / "cir_video.mp4"
mask_video_rgb_path = output_dir / "mask_propagation_rgb.mp4"
mask_video_cir_path = output_dir / "mask_propagation_cir.mp4"

passive_cu3s = (
    dataset_dir
    / "XMR_Demo_Object_Tracking"
    / "measurements"
    / "cu3s"
    / "2026_04_15_16_28_10"
    / "Auto_000.cu3s"
)

print(f"Passive CU3S:        {passive_cu3s}")
print(f"Text prompt:         {TEXT_PROMPT}")
print(f"Mask prompt spec:    {MASK_PROMPT_SPEC}")
print(f"Tracking overlay:    {output_video_path}")
print(f"Clean RGB video:     {rgb_video_path}")
print(f"Tracking JSON:       {output_json_path}")
print(f"CIR video:           {cir_video_path}")
print(f"Mask-prop RGB video: {mask_video_rgb_path}")
print(f"Mask-prop CIR video: {mask_video_cir_path}")

## 4 · Load the SAM3 plugin

SAM3 is a plugin shipped separately from the Cuvis.AI core. `NodeRegistry` resolves
its package, downloads and caches the model weights on first use, and returns the
node class ready to instantiate like any built-in node.

The plugin manifest at `configs/plugins/sam3.yaml` lists the Git tag and provides
class names for all SAM3 node variants
(`SAM3TextPropagation`, `SAM3MaskPropagation`, `SAM3BboxPropagation`, …).

In [ ]:
PLUGINS_YAML = Path("../../configs/plugins/sam3.yaml")

registry = NodeRegistry()
registry.load_plugins(str(PLUGINS_YAML))
SAM3TextPropagation = registry.get("cuvis_ai_sam3.node.SAM3TextPropagation")

logger.success("SAM3TextPropagation loaded: {}", SAM3TextPropagation)

## 5 · Build the tracking pipeline

Eight nodes, sixteen connections:

- `CU3SDataNode` unpacks each measurement into a `[B, H, W, C]` spectral cube,
  a 1-D wavelength vector, and a measurement index (`mesu_index`).
- `CIETristimulusRGBSelector` integrates the cube against CIE colour-matching
  functions to produce a normalised `[B, H, W, 3]` RGB frame.
- `TextPrompt` schedules text prompts at specific frame IDs; `"person@0"` emits
  `"person"` once at frame 0 to seed the tracker and nothing on subsequent frames.
- `SAM3TextPropagation` is stateful: it segments the prompt targets on the seeded
  frame and propagates the masks forward through the video stream.
- `TrackingOverlayNode` blends the predicted masks and object IDs onto the
  RGB frame as coloured outlines.
- `ToVideoNode` (overlay) encodes the annotated frames to `tracking_overlay.mp4`.
- `ToVideoNode` (clean RGB) writes the un-annotated CIE-tristimulus RGB to
  `rgb_video.mp4`. The mask-propagation pipelines in §10 read this file directly
  rather than re-decoding the cu3s cube.
- `CocoTrackMaskWriter` records per-frame SAM3 detections to
  `tracking_results.json` so §10–11 can seed `MaskPrompt` from one of those
  detections.

In [ ]:
pipeline = CuvisPipeline("HyperspectralTracking_SAM3_Text")

cu3s_data = CU3SDataNode(name="cu3s_data")
true_rgb = CIETristimulusRGBSelector(name="true_rgb")
text_prompt = TextPrompt(prompt_specs=[TEXT_PROMPT], name="text_prompt")
sam3_tracker = SAM3TextPropagation(
    checkpoint_path=None,
    compile_model=False,
    score_threshold_detection=0.5,
    new_det_thresh=0.7,
    det_nms_thresh=0.1,
    overlap_suppress_thresh=0.7,
    max_tracker_states=500,
    name="sam3_tracker",
)
overlay = TrackingOverlayNode(alpha=0.2, name="tracking_overlay")
to_video = ToVideoNode(
    output_video_path=str(output_video_path),
    frame_rate=FRAME_RATE,
    name="to_video",
)
to_rgb_video = ToVideoNode(
    output_video_path=str(rgb_video_path),
    frame_rate=FRAME_RATE,
    name="to_rgb_video",
)
tracking_json = CocoTrackMaskWriter(
    output_json_path=str(output_json_path),
    default_category_name="person",
    name="tracking_coco_json",
)

pipeline.connect(
    # Hyperspectral source → RGB
    (cu3s_data.outputs.cube, true_rgb.cube),
    (cu3s_data.outputs.wavelengths, true_rgb.wavelengths),
    # RGB + frame ID → SAM3 tracker
    (true_rgb.rgb_image, sam3_tracker.rgb_frame),
    (cu3s_data.outputs.mesu_index, sam3_tracker.inputs.frame_id),
    # Text prompt schedule → SAM3 tracker
    (cu3s_data.outputs.mesu_index, text_prompt.frame_id),
    (text_prompt.text_prompt, sam3_tracker.inputs.text_prompt),
    # SAM3 outputs + RGB → overlay
    (true_rgb.rgb_image, overlay.rgb_image),
    (cu3s_data.outputs.mesu_index, overlay.frame_id),
    (sam3_tracker.mask, overlay.mask),
    (sam3_tracker.object_ids, overlay.object_ids),
    # Overlay → tracking video
    (overlay.rgb_with_overlay, to_video.rgb_image),
    (cu3s_data.outputs.mesu_index, to_video.frame_id),
    # Clean RGB → rgb_video.mp4 (consumed by §10)
    (true_rgb.rgb_image, to_rgb_video.rgb_image),
    (cu3s_data.outputs.mesu_index, to_rgb_video.frame_id),
    # SAM3 outputs → COCO JSON (consumed by §10–11 MaskPrompt)
    (cu3s_data.outputs.mesu_index, tracking_json.frame_id),
    (sam3_tracker.mask, tracking_json.mask),
    (sam3_tracker.object_ids, tracking_json.object_ids),
    (sam3_tracker.detection_scores, tracking_json.detection_scores),
)

In [ ]:
graph_png = pipeline.visualize(
    format="render_graphviz",
    output_path=str(output_dir / f"{pipeline.name}.png"),
)
display(Image(str(graph_png)))

## 6 · Sanity-check: one RGB frame

Before committing to the full sweep, render a single frame end-to-end to verify
the RGB colours, orientation, and overall sanity — takes a few seconds.
`collect_outputs=True` captures intermediate tensors keyed by
`(node_name, port_name)` so we can pull and plot the RGB image inline.

*Note:* `ToVideoNode` writes a one-frame MP4 on this preview run; §7 overwrites
it on the full sweep.

In [ ]:
preview_datamodule = SingleCu3sDataModule(
    cu3s_file_path=str(passive_cu3s),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    predict_ids=[PREVIEW_FRAME_ID],
)

pipeline.to(torch.device("cpu"))
preview_outputs = Predictor(pipeline=pipeline, datamodule=preview_datamodule).predict(
    collect_outputs=True
)

rgb_frame = preview_outputs[0][("true_rgb", "rgb_image")][0].cpu().numpy()

plt.figure(figsize=(9, 7))
plt.imshow(rgb_frame)
plt.title(f"RGB (CIE tristimulus) — passive session, frame {PREVIEW_FRAME_ID}")
plt.axis("off")
plt.show()

## 7 · Run the pipeline

`Predictor.predict` iterates batches from `SingleCu3sDataModule` and pushes each
through the connected pipeline. This run is capped at **200 frames** —
`max_batches=200` — because SAM3 text propagation is the slowest step and 200
frames (~13 s at 15 fps) is enough to show identity preservation through the
first occlusion. Drop the `max_batches` argument to process the full session.

SAM3 is stateful: the tracker seeds on the first frame where the text prompt
matches, then propagates forward. Re-instantiating the SAM3 node (or reconstructing
the pipeline) resets this state — useful if you want to run on a different clip.

In [ ]:
datamodule = SingleCu3sDataModule(
    cu3s_file_path=str(passive_cu3s),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    predict_ids=None,
)

pipeline.to(torch.device("cpu"))
predictor = Predictor(pipeline=pipeline, datamodule=datamodule)
predictor.predict(max_batches=90, collect_outputs=False)

for path in (output_video_path, rgb_video_path, output_json_path):
    if not path.exists():
        raise RuntimeError(f"Expected output was not created: {path}")

logger.success("Tracking overlay:    {}", output_video_path)
logger.success("Clean RGB video:     {}", rgb_video_path)
logger.success("Tracking COCO JSON:  {}", output_json_path)

## 8 · Watch the result

In [ ]:
display(Video(str(output_video_path), embed=False, width=640))

## 9 · CIR false-RGB rendering

CIR ("Color Infrared") is a false-RGB rendering of the hyperspectral cube where the
**near-infrared** band drives the red channel, **red** drives green, and **green** drives blue:

| Output channel | Source band  |
|----------------|--------------|
| R              | NIR ≈ 860 nm |
| G              | Red ≈ 670 nm |
| B              | Green ≈ 560 nm |

Vegetation reflects strongly in NIR and so glows bright red. More importantly for
this tutorial, garments that look identical in true RGB often differ markedly in NIR —
exactly the property that lets SAM3 mask propagation lock onto the right person in §11.

Like the passive section, we follow the *prepare data → build pipeline → run pipeline*
rhythm. The pipeline is `CU3SDataNode → CIRSelector → ToVideoNode`.

### 9.1 Build the CIR pipeline

```
CU3SDataNode ──► CIRSelector ──► ToVideoNode
```

In [ ]:
cir_pipeline = CuvisPipeline("CIR_FalseRGB")

cir_cu3s_data = CU3SDataNode(name="cu3s_data")
cir_selector = CIRSelector(nir_nm=860.0, red_nm=670.0, green_nm=560.0, name="cir_rgb")
cir_to_video = ToVideoNode(
    output_video_path=str(cir_video_path),
    frame_rate=FRAME_RATE,
    name="to_video",
)

cir_pipeline.connect(
    (cir_cu3s_data.outputs.cube, cir_selector.cube),
    (cir_cu3s_data.outputs.wavelengths, cir_selector.wavelengths),
    (cir_selector.rgb_image, cir_to_video.rgb_image),
    (cir_cu3s_data.outputs.mesu_index, cir_to_video.frame_id),
)

cir_graph_png = cir_pipeline.visualize(
    format="render_graphviz",
    output_path=str(output_dir / f"{cir_pipeline.name}.png"),
)
display(Image(str(cir_graph_png)))

### 9.2 Preview a single frame

Render frame `PREVIEW_FRAME_ID` and place it next to the CIE-tristimulus rendering
from §6 — vegetation should glow red and any NIR-reflective fabric should pop out
relative to its true-RGB appearance.

In [ ]:
cir_preview_datamodule = SingleCu3sDataModule(
    cu3s_file_path=str(passive_cu3s),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    predict_ids=[PREVIEW_FRAME_ID],
)

cir_pipeline.to(torch.device("cpu"))
cir_preview_outputs = Predictor(pipeline=cir_pipeline, datamodule=cir_preview_datamodule).predict(
    collect_outputs=True
)

cir_frame = cir_preview_outputs[0][("cir_rgb", "rgb_image")][0].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(rgb_frame)
axes[0].set_title(f"True RGB (CIE tristimulus) — frame {PREVIEW_FRAME_ID}")
axes[0].axis("off")
axes[1].imshow(cir_frame)
axes[1].set_title(f"CIR (NIR→R, R→G, G→B) — frame {PREVIEW_FRAME_ID}")
axes[1].axis("off")
plt.suptitle("True RGB vs. CIR — passive session", y=1.01)
plt.tight_layout()
plt.show()

### 9.3 Render the full CIR video

Re-run the same pipeline over every frame to write `cir_video.mp4`. This file
becomes the input video for §11's mask-propagation pipeline.

In [ ]:
cir_datamodule = SingleCu3sDataModule(
    cu3s_file_path=str(passive_cu3s),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    predict_ids=None,
)

cir_pipeline.to(torch.device("cpu"))
Predictor(pipeline=cir_pipeline, datamodule=cir_datamodule).predict(collect_outputs=False)

if not cir_video_path.exists():
    raise RuntimeError(f"CIR video was not created: {cir_video_path}")

logger.success("CIR video: {}", cir_video_path)

### 9.4 Watch the CIR video

In [ ]:
display(Video(str(cir_video_path), embed=False, width=640))

## 10 · Mask propagation on the RGB video

`SAM3TextPropagation` (used in §5–8) tracks **everyone** the text prompt matches —
class-level tracking. To lock onto **one specific person** instead, we use
`SAM3MaskPropagation`: seed it with a single mask on a single frame, and it
propagates that mask through the rest of the video.

Two ingredients we already have from §7:

- `rgb_video.mp4` — the clean CIE-tristimulus RGB rendering of every frame.
- `tracking_results.json` — per-frame SAM3 detections in COCO format. We pick one
  detection from this file as the seed mask, via `MASK_PROMPT_SPEC` (§3).

The pipeline reads frames from the MP4, so it bypasses the cu3s decode entirely.

### 10.1 Prepare data

The pipeline used here:

```
VideoFrameNode ──► SAM3MaskPropagation ──► TrackingOverlayNode ──► ToVideoNode
                          ▲
                     MaskPrompt
                          ▲
                tracking_results.json
```

We could equally well decode the cu3s cube on the fly — the alternative pipeline:

```
CU3SDataNode ──► CIETristimulusRGBSelector ──► SAM3MaskPropagation ──► TrackingOverlayNode ──► ToVideoNode
                                                       ▲
                                                  MaskPrompt
```

Both forms produce the same overlay; the cu3s form re-renders the false-RGB on
every run, the video form reuses the cached MP4 we already wrote in §7. We use
the video form here.

In [ ]:
for path in (rgb_video_path, output_json_path):
    if not path.exists():
        raise RuntimeError(f"Missing prerequisite from §7: {path}. Run §5–7 before this section.")

print(f"Source video:     {rgb_video_path}")
print(f"Detection JSON:   {output_json_path}")
print(f"Mask prompt spec: {MASK_PROMPT_SPEC}  (object_id : detection_id @ frame_id)")

### 10.2 Build the mask propagation pipeline

In [ ]:
SAM3MaskPropagation = registry.get("cuvis_ai_sam3.node.SAM3MaskPropagation")

mask_pipeline_rgb = CuvisPipeline("MaskPropagation_RGB_Video")

video_frame_rgb = VideoFrameNode(name="video_frame")
mask_prompt_rgb = MaskPrompt(
    json_path=str(output_json_path),
    prompt_specs=[MASK_PROMPT_SPEC],
    name="mask_prompt",
)
sam3_mask_rgb = SAM3MaskPropagation(
    checkpoint_path=None,
    compile_model=False,
    score_threshold_detection=0.5,
    new_det_thresh=0.7,
    det_nms_thresh=0.1,
    overlap_suppress_thresh=0.7,
    max_tracker_states=5,
    name="sam3_mask",
)
overlay_rgb = TrackingOverlayNode(alpha=0.2, name="tracking_overlay")
to_video_rgb = ToVideoNode(
    output_video_path=str(mask_video_rgb_path),
    frame_rate=FRAME_RATE,
    name="to_video",
)

mask_pipeline_rgb.connect(
    # Video source → SAM3 tracker
    (video_frame_rgb.outputs.rgb_image, sam3_mask_rgb.rgb_frame),
    (video_frame_rgb.outputs.frame_id, sam3_mask_rgb.inputs.frame_id),
    # Mask prompt → SAM3 tracker
    (video_frame_rgb.outputs.frame_id, mask_prompt_rgb.frame_id),
    (mask_prompt_rgb.mask, sam3_mask_rgb.inputs.mask),
    # SAM3 outputs + RGB → overlay
    (video_frame_rgb.outputs.rgb_image, overlay_rgb.rgb_image),
    (video_frame_rgb.outputs.frame_id, overlay_rgb.frame_id),
    (sam3_mask_rgb.outputs.mask, overlay_rgb.mask),
    (sam3_mask_rgb.outputs.object_ids, overlay_rgb.object_ids),
    # Overlay → video
    (overlay_rgb.rgb_with_overlay, to_video_rgb.rgb_image),
    (video_frame_rgb.outputs.frame_id, to_video_rgb.frame_id),
)

In [ ]:
mask_graph_rgb_png = mask_pipeline_rgb.visualize(
    format="render_graphviz",
    output_path=str(output_dir / f"{mask_pipeline_rgb.name}.png"),
)
display(Image(str(mask_graph_rgb_png)))

### 10.3 Run the mask propagation

In [ ]:
mask_datamodule_rgb = VideoFrameDataModule(
    video_path=str(rgb_video_path),
    end_frame=-1,
    batch_size=1,
)

mask_pipeline_rgb.to(torch.device("cpu"))
Predictor(pipeline=mask_pipeline_rgb, datamodule=mask_datamodule_rgb).predict(collect_outputs=False)

if not mask_video_rgb_path.exists():
    raise RuntimeError(f"Mask-prop video was not created: {mask_video_rgb_path}")

logger.success("RGB mask-prop overlay: {}", mask_video_rgb_path)

### 10.4 Watch the mask-propagation overlay

In [ ]:
display(Video(str(mask_video_rgb_path), embed=False, width=640))

## 11 · Mask propagation on the CIR video

Same `SAM3MaskPropagation` setup as §10, but the input video is the CIR rendering
from §9 instead of the true-RGB one. The seed comes from the same
`tracking_results.json` and the same `MASK_PROMPT_SPEC` — frame indices in the
JSON are aligned to source measurements, so the same prompt picks the same person
in either rendering.

Why bother running mask propagation on CIR? When SAM3's appearance features
struggle on the true-RGB stream (visually similar clothing, low contrast), the
NIR-driven CIR rendering can give the tracker more spectral signal to lock onto.

### 11.1 Prepare data

In [ ]:
for path in (cir_video_path, output_json_path):
    if not path.exists():
        raise RuntimeError(
            f"Missing prerequisite: {path}. Run §9 (CIR video) and §5–7 (JSON) first."
        )

print(f"Source video:     {cir_video_path}")
print(f"Detection JSON:   {output_json_path}")
print(f"Mask prompt spec: {MASK_PROMPT_SPEC}")

### 11.2 Build the mask propagation pipeline

A fresh `SAM3MaskPropagation` instance — the tracker is stateful, so reusing the
§10 instance would carry over its propagation state.

In [ ]:
mask_pipeline_cir = CuvisPipeline("MaskPropagation_CIR_Video")

video_frame_cir = VideoFrameNode(name="video_frame")
mask_prompt_cir = MaskPrompt(
    json_path=str(output_json_path),
    prompt_specs=[MASK_PROMPT_SPEC],
    name="mask_prompt",
)
sam3_mask_cir = SAM3MaskPropagation(
    checkpoint_path=None,
    compile_model=False,
    score_threshold_detection=0.5,
    new_det_thresh=0.7,
    det_nms_thresh=0.1,
    overlap_suppress_thresh=0.7,
    max_tracker_states=5,
    name="sam3_mask",
)
overlay_cir = TrackingOverlayNode(alpha=0.2, name="tracking_overlay")
to_video_cir = ToVideoNode(
    output_video_path=str(mask_video_cir_path),
    frame_rate=FRAME_RATE,
    name="to_video",
)

mask_pipeline_cir.connect(
    (video_frame_cir.outputs.rgb_image, sam3_mask_cir.rgb_frame),
    (video_frame_cir.outputs.frame_id, sam3_mask_cir.inputs.frame_id),
    (video_frame_cir.outputs.frame_id, mask_prompt_cir.frame_id),
    (mask_prompt_cir.mask, sam3_mask_cir.inputs.mask),
    (video_frame_cir.outputs.rgb_image, overlay_cir.rgb_image),
    (video_frame_cir.outputs.frame_id, overlay_cir.frame_id),
    (sam3_mask_cir.outputs.mask, overlay_cir.mask),
    (sam3_mask_cir.outputs.object_ids, overlay_cir.object_ids),
    (overlay_cir.rgb_with_overlay, to_video_cir.rgb_image),
    (video_frame_cir.outputs.frame_id, to_video_cir.frame_id),
)

mask_graph_cir_png = mask_pipeline_cir.visualize(
    format="render_graphviz",
    output_path=str(output_dir / f"{mask_pipeline_cir.name}.png"),
)
display(Image(str(mask_graph_cir_png)))

### 11.3 Run the mask propagation

In [ ]:
mask_datamodule_cir = VideoFrameDataModule(
    video_path=str(cir_video_path),
    end_frame=-1,
    batch_size=1,
)

mask_pipeline_cir.to(torch.device("cpu"))
Predictor(pipeline=mask_pipeline_cir, datamodule=mask_datamodule_cir).predict(collect_outputs=False)

if not mask_video_cir_path.exists():
    raise RuntimeError(f"Mask-prop video was not created: {mask_video_cir_path}")

logger.success("CIR mask-prop overlay: {}", mask_video_cir_path)

### 11.4 Watch the mask-propagation overlay

In [ ]:
display(Video(str(mask_video_cir_path), embed=False, width=640))

## 12 · Active tracking with invisible spectral ink

The second dataset session (`2026_04_20`) demonstrates a fundamentally different
tracking mode: instead of exploiting pre-existing spectral differences between
garments, one actor **creates** a spectral signature mid-scene by spraying a
target with near-infrared-fluorescent ink that is completely invisible to the
naked eye and to any RGB camera.

### How Spectral Angle Mapper (SAM) tracking works

$$
\theta(\mathbf{p}, \mathbf{r}) =
  \arccos\!\left(
    \frac{\mathbf{p} \cdot \mathbf{r}}{\|\mathbf{p}\|\,\|\mathbf{r}\|}
  \right)
$$

where $\mathbf{p} \in \mathbb{R}^{C}$ is a pixel's spectrum and
$\mathbf{r} \in \mathbb{R}^{C}$ is a reference spectrum of the ink.
A small angle ($\theta \approx 0$) means the pixel's spectral shape closely
matches the ink signature, regardless of overall brightness.

The full tracking pipeline for the active session is:

```
CU3SDataNode ──► BandpassByWavelength ──► SpectralAngleMapper ──► BinaryDecider ──► MaskRobustifier ──► MaskOverlayNode ──► ToVideoNode
                                   ▲              ▲
                              NpyReader ──────────┘
                          (reference spectrum)
```

Key properties vs. SAM3 mask propagation:

| Property | SAM3 mask propagation | SpectralAngleMapper |
|----------|-----------------------|---------------------|
| Requires training / retraining | No | No |
| Reference data needed | One seed mask | 1 spectrum vector |
| Works without ink | Yes | No (needs a marked target) |
| Fails on visually ambiguous targets | Sometimes | No (spectral, not appearance) |
| Robust to occlusion | Drift possible | Re-acquires on reappearance |

The built-in node is `cuvis_ai.node.spectral_angle_mapper.SpectralAngleMapper`.
The full runnable script — including bandpass filtering, Kalman-filtered bounding
box, and zoom-crop output — lives in the
[cuvis-ai-cookbook](https://github.com/cubert-hyperspectral/cuvis-ai-cookbook/blob/main/examples/spectral_angle_mapper/spam_invisible_ink.py)
repo.

### Preview: active session RGB

The cell below loads a single frame from the active session and shows the RGB
image alongside the passive session for visual comparison. The reference spectrum
`.npy` file required to run the SPAM pipeline is produced in a separate calibration
step (see the script's `--reference-npy` argument).

### 12.1 Prepare data

In [ ]:
active_cu3s = (
    dataset_dir
    / "XMR_Demo_Object_Tracking"
    / "measurements"
    / "cu3s"
    / "2026_04_20_16_28_54"
    / "Auto_000.cu3s"
)

active_preview_dir = output_dir / "active_preview"
active_preview_dir.mkdir(parents=True, exist_ok=True)

active_video_path = active_preview_dir / "active_rgb.mp4"

print(f"Active CU3S:       {active_cu3s}")
print(f"Active output dir: {active_preview_dir}")
print(f"Active video path: {active_video_path}")

### 12.2 Build the active-session pipeline

Same `CU3SDataNode → CIETristimulusRGBSelector → ToVideoNode` shape as §9 — only
the source cu3s file changes. Visualize the graph to confirm the wiring.

In [ ]:
active_pipeline = CuvisPipeline("ActiveSession_RGB")

active_cu3s_node = CU3SDataNode(name="cu3s_data")
active_true_rgb = CIETristimulusRGBSelector(name="true_rgb")
active_to_video = ToVideoNode(
    output_video_path=str(active_video_path),
    frame_rate=FRAME_RATE,
    name="to_video",
)

active_pipeline.connect(
    (active_cu3s_node.outputs.cube, active_true_rgb.cube),
    (active_cu3s_node.outputs.wavelengths, active_true_rgb.wavelengths),
    (active_true_rgb.rgb_image, active_to_video.rgb_image),
    (active_cu3s_node.outputs.mesu_index, active_to_video.frame_id),
)

active_graph_png = active_pipeline.visualize(
    format="render_graphviz",
    output_path=str(active_preview_dir / f"{active_pipeline.name}.png"),
)
display(Image(str(active_graph_png)))

### 12.3 Compare a single frame

Render frame `PREVIEW_FRAME_ID` from the active session and place it next to the
same frame from the passive session — the two should look indistinguishable in
true RGB even though one of the actors will soon be marked with NIR-fluorescent
ink that any RGB sensor (including this one) cannot see.

In [ ]:
active_preview_datamodule = SingleCu3sDataModule(
    cu3s_file_path=str(active_cu3s),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    predict_ids=[PREVIEW_FRAME_ID],
)

active_pipeline.to(torch.device("cpu"))
active_outputs = Predictor(pipeline=active_pipeline, datamodule=active_preview_datamodule).predict(
    collect_outputs=True
)

active_frame = active_outputs[0][("true_rgb", "rgb_image")][0].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(rgb_frame)
axes[0].set_title(f"Passive session — frame {PREVIEW_FRAME_ID}")
axes[0].axis("off")
axes[1].imshow(active_frame)
axes[1].set_title(f"Active session — frame {PREVIEW_FRAME_ID}")
axes[1].axis("off")
plt.suptitle("RGB (CIE tristimulus): passive vs. active sessions", y=1.01)
plt.tight_layout()
plt.show()

### 12.4 Render the full active video

In [ ]:
active_datamodule = SingleCu3sDataModule(
    cu3s_file_path=str(active_cu3s),
    processing_mode=PROCESSING_MODE,
    batch_size=1,
    predict_ids=None,
)

active_pipeline.to(torch.device("cpu"))
Predictor(pipeline=active_pipeline, datamodule=active_datamodule).predict(collect_outputs=False)

if not active_video_path.exists():
    raise RuntimeError(f"Active video was not created: {active_video_path}")

logger.success("Active session video: {}", active_video_path)

### 12.5 Watch the active session video

In [ ]:
display(Video(str(active_video_path), embed=False, width=640))